# 第1回 演習: 関数とデコレータ

関数を書くところから始めて、関数を辞書に入れたり、関数を包んだりする。
課題1 の遅い再帰に、課題3 の `@memo` を付けると一気に速くなる。
課題4 では、課題2 の電卓を文字列から呼べるようにする。

## 全体の地図

| 課題 | 書くもの | 学ぶこと | 配点 |
|---|---|---|---|
| 課題0 | `greet`, `apply_twice` | 引数、デフォルト値、関数を引数に渡す | 10 |
| 課題1 | `tarai` | 再帰、呼び出し回数を数える | 10 |
| 課題1-b | `distinct_args` | なぜ遅いのか (同じ引数の繰り返し) | 10 |
| 課題2 | `OPS`, `calc` | 辞書に関数を入れる | 15 |
| 課題3 | `memo` | デコレータ (関数を包む) | 15 |
| 課題3-b | `tarai_memo` | `@memo` を課題1 に付ける | 10 |
| 課題4 | `command`, `run` | 文字列から関数を実行する | 20 |
| 発展1・2 | `trace`, `help_text` | 余力があれば | 各5 |

## 進め方

- 上から順に進める。各課題は「例題セルを実行 → `TODO` を埋める → 確認セルを実行」
- 例題セルは動きを見るためのもの。そのまま実行すればよい
- 確認セル (`# [CHECK]`) は自分用。消しても教員の採点は別途走るが、残すこと
- 終わったら出力を残したまま保存し、kibaco から提出する

In [ ]:
#@title 自動セットアップ (実行するだけ。中身は隠しています)
# [SKIP] 採点の対象外
# ---- 自己確認の準備 (check / score)。1回実行すればよい ----
import pathlib
import urllib.request

if not pathlib.Path('autograde.py').exists():
    try:
        urllib.request.urlretrieve(
            'https://3tty0n.github.io/2026-scripting-lab/grading/autograde.py', 'autograde.py')
    except Exception as e:
        print('取得できませんでした:', e)

from autograde import check, score
TASK = '01'


---
## 課題0: 関数を書く・渡す (10点)

まずは基本。2つの関数を書け。

- `greet(name, suffix='!')`: `'Hello, ' + name + suffix` を返す。`suffix` は省略できる
- `apply_twice(func, x)`: 関数 `func` を2回かけた結果 `func(func(x))` を返す

`apply_twice` は、**関数を引数として受け取る**練習。
この後の課題 (辞書に関数を入れる、デコレータ) の土台になる。

**ヒント**: 関数も値なので、`func` という名前の引数に入っている。呼ぶときは `func(x)`。

### 例題

次のセルを実行して、動きを見る。

デフォルト値つきの引数と、関数を引数に渡す形。

In [ ]:
# 例題
def area(w, h=1):
    return w * h


print(area(3, 4), area(3))               # 12 3


def call(f, x):
    return f(x)


print(call(len, 'abc'), call(abs, -5))   # 3 5

In [ ]:
def greet(name, suffix='!'):
    # TODO: 'Hello, ' + name + suffix を返す
    pass


def apply_twice(func, x):
    # TODO: func を2回かけた結果を返す
    pass


print(greet('Alice'), greet('Bob', '.'))
print(apply_twice(abs, -3), apply_twice(str.upper, 'ab'))

In [ ]:
# [CHECK] q0
need('greet', arity=2)
need('apply_twice', arity=2)

assert greet('Alice') == 'Hello, Alice!'
assert greet('Bob', '.') == 'Hello, Bob.'
assert greet(name='Eve', suffix='?') == 'Hello, Eve?', '名前付きの引数でも呼べること'
assert apply_twice(lambda x: x + 1, 0) == 2
assert apply_twice(lambda x: x * 3, 2) == 18
assert apply_twice(str.upper, 'ab') == 'AB'
print('OK')

---
## 課題1: 竹内関数 (10点)

竹内関数 (たらいまわし関数) は、値そのものは単純なのに
**呼び出し回数が爆発する**ことで知られる再帰関数。

```
tarai(x, y, z) = y                                (x <= y のとき)
               = tarai(tarai(x-1, y, z),
                       tarai(y-1, z, x),
                       tarai(z-1, x, y))          (それ以外)
```

実装し、**何回呼ばれたか**を数えよ。

**ヒント**: 関数の中からグローバル変数を書き換えるには `global CALLS` の宣言が要る。

### 例題

次のセルを実行して、動きを見る。

同じ形の、もっと簡単な再帰。数え方はこれと同じ。


In [ ]:
# 例題
FIB_CALLS = 0


def fib(n):
    global FIB_CALLS
    FIB_CALLS += 1
    if n <= 1:
        return n
    return fib(n - 1) + fib(n - 2)


FIB_CALLS = 0
print(fib(10), FIB_CALLS)    # 55 177

`tarai` も形は同じ。止まる条件が `x <= y`、
そうでなければ `tarai` を3回呼んだ結果をさらに `tarai` に渡す。


In [ ]:
CALLS = 0


def tarai(x, y, z):
    global CALLS
    CALLS += 1
    if x <= y:
        return ...      # TODO: 止まるときの値
    return ...          # TODO: tarai を3回呼んだ結果で、もう一度 tarai を呼ぶ


import time
CALLS = 0
t0 = time.time()
print('tarai(12, 6, 0) =', tarai(12, 6, 0))
print(f'呼び出し回数: {CALLS:,} 回 / {time.time() - t0:.2f} 秒')

In [ ]:
# [CHECK] q1
need('tarai', arity=3)
need('CALLS', kind=int)

CALLS = 0; assert tarai(1, 2, 3) == 2      # x <= y なので y
CALLS = 0; assert tarai(3, 2, 1) == 3
# 自己申告の CALLS だけに頼らず、外から呼び出しそのものを観察する
CALLS = 0
with watch('tarai') as spy:
    assert tarai(10, 5, 0) == 10
assert CALLS == 343073, f'呼び出し回数が {CALLS} 回。数え方を見直すこと'
assert spy.calls == CALLS, f'外から数えると {spy.calls:,} 回。数え方がずれている'
note(f'tarai(10, 5, 0) の呼び出し {spy.calls:,} 回')

print('OK')


### 課題1-b: なぜ増えるのか (10点)

`tarai(12, 6, 0)` は 1,260万回も呼ばれる。しかし引数は3つとも小さい整数で、
**同じ引数の組が何度も現れている**。

`distinct_args(x, y, z)` を書け。`tarai(x, y, z)` を計算する過程で現れる
**異なる引数の組 `(x, y, z)` が何通りあるか**を返す。

**ヒント**: 集合に `(x, y, z)` のタプルを入れていき、最後に `len()` を取る。
`tarai` とは別に、記録用の関数を書くとよい。

**注意**: `tarai(14, 7, 0)` は5億回を超える。試さないこと。

### 例題

次のセルを実行して、動きを見る。

`fib` について、同じことをやった例。


In [ ]:
# 例題
def distinct_fib(n):
    seen = set()

    def walk(n):
        seen.add(n)
        if n <= 1:
            return n
        return walk(n - 1) + walk(n - 2)

    walk(n)
    return len(seen)


print(distinct_fib(10))    # 11 (177 回呼ばれても、引数は 11 通り)

`tarai` は引数が3つなので、記録するのは `seen.add((x, y, z))` とタプルにする。
`walk` の中身は課題1 の `tarai` と同じ計算。ただし呼ぶ相手は `walk` 自身。


In [ ]:
def distinct_args(x, y, z):
    seen = set()

    def walk(x, y, z):
        # TODO: seen に記録しながら、tarai と同じ計算をする
        pass

    walk(x, y, z)
    return len(seen)


print('tarai(10, 5, 0):', f'{343073:,} 回の呼び出し /',
      distinct_args(10, 5, 0), '通りの引数')
print('tarai(12, 6, 0):', f'{12604861:,} 回の呼び出し /',
      distinct_args(12, 6, 0), '通りの引数')

In [ ]:
# [CHECK] q1b
need('distinct_args', arity=3)

assert distinct_args(1, 2, 3) == 1        # 1回で終わる
assert distinct_args(10, 5, 0) == 157, distinct_args(10, 5, 0)
assert distinct_args(12, 6, 0) == 225, distinct_args(12, 6, 0)
print('OK  1,260万回の呼び出しに対して、引数の組はたった 225 通り')


---
## 課題2: 辞書に関数を入れて電卓 (15点)

`if` を並べずに、**辞書に関数を入れて**電卓を作る。

- `OPS`: 演算子の文字列から「2引数の関数」を引く辞書
- `calc(op, a, b)`: `OPS` を引いて呼ぶ。**本体は1行**
- 知らない演算子なら `ValueError`

**ヒント**: `operator` モジュールに `add`, `sub`, `mul`, `floordiv`, `mod`, `pow` がある。
`lambda` を自分で書いてもよい。

### 例題

次のセルを実行して、動きを見る。

辞書に関数を入れて、名前で引いて呼ぶ形。

In [ ]:
# 例題
import math

FUNCS = {
    'sqrt': math.sqrt,
    'abs': abs,
}


def apply1(name, x):
    if name not in FUNCS:
        raise ValueError(f'知らない関数: {name}')
    return FUNCS[name](x)


print(apply1('sqrt', 16), apply1('abs', -3))   # 4.0 3

`calc` はこれの2引数版。`operator.add(3, 4)` は `3 + 4` と同じ。


In [ ]:
import operator

OPS = {
    '+': operator.add,
    # TODO: - * / % ** を追加
}


def calc(op, a, b):
    # TODO: 知らない演算子は ValueError。そうでなければ OPS から関数を引いて呼ぶ
    pass


print(calc('+', 3, 4), calc('*', 3, 4), calc('**', 2, 10))

In [ ]:
# [CHECK] q2
need('OPS', kind=dict)
need('calc', arity=3)

assert calc('+', 3, 4) == 7
assert calc('-', 10, 3) == 7
assert calc('*', 3, 4) == 12
assert calc('/', 20, 6) == 3        # 整数除算にすること
assert calc('%', 20, 6) == 2
assert calc('**', 2, 10) == 1024
try:
    calc('?', 1, 2)
except ValueError:
    pass
else:
    raise AssertionError('知らない演算子は ValueError を出すこと')

# 実行中に演算子を増やせること (動的言語らしさ)
OPS['max'] = max
assert calc('max', 3, 9) == 9

# 具体例だけでなく、成り立つべき性質でも確かめる
holds('a + b は b + a と同じ', lambda a, b: calc('+', a, b) == calc('+', b, a),
      ints(2))
holds('a - b は -(b - a) と同じ', lambda a, b: calc('-', a, b) == -calc('-', b, a),
      ints(2))
holds('a * 1 は a のまま', lambda a: calc('*', a, 1) == a, ints(1))
# 書き方も見る。OPS を作っただけで if / elif に戻っていないこと
_t = tags('calc')
assert not _t or ({'subscript', 'calls:get'} & _t), \
    'OPS を作っても calc が if / elif で分けている。表から引いて呼ぶこと'

print('OK')


---
## 課題3: `@memo` デコレータ (15点)

**関数を受け取って、結果をキャッシュする関数を返す**デコレータを書け。

```python
def memo(func):
    cache = {}
    def wrapper(*args):
        ...             # cache に無ければ計算して入れる
    return wrapper
```

**ヒント**: `*args` は引数を**タプル**にまとめる。タプルは辞書のキーにできる。

### 例題

次のセルを実行して、動きを見る。

まず、デコレータを使わずに `fib` だけをキャッシュした例。

In [ ]:
# 例題
fib_cache = {}


def fib_fast(n):
    if n in fib_cache:
        return fib_cache[n]
    r = n if n <= 1 else fib_fast(n - 1) + fib_fast(n - 2)
    fib_cache[n] = r
    return r


print(fib_fast(100))    # 一瞬で返る

課題は、この仕組みを**どんな関数にも使えるように**外に出すこと。
`func` を受け取り、上と同じ判定をする `wrapper` を作って返す。
キーになるのは `n` ではなく引数全部のタプル `args`。


In [ ]:
# 例題
def counted(func):
    def wrapper(*args):
        print('呼ばれた:', args)
        return func(*args)
    return wrapper


@counted
def add(a, b):
    return a + b


add(1, 2)    # 呼ばれた: (1, 2)

In [ ]:
def memo(func):
    cache = {}

    def wrapper(*args):
        # TODO: args が cache に無ければ計算して入れる。cache[args] を返す
        pass

    wrapper.cache = cache
    return wrapper

In [ ]:
# [CHECK] q3
need('memo', arity=1)

calls = 0

@memo
def slow_add(a, b):
    global calls
    calls += 1
    return a + b

assert slow_add(1, 2) == 3
assert slow_add(1, 2) == 3
assert slow_add(1, 2) == 3
assert calls == 1, f'3回呼んだのに本体が {calls} 回実行されている'
assert slow_add(2, 3) == 5
assert calls == 2, '引数が違えば計算し直すこと'
assert len(slow_add.cache) == 2

# 引数の個数が違う関数にも使えること
@memo
def one(x):
    return x * 2
assert one(5) == 10 and one(5) == 10
print('OK')


### 課題3-b: 課題1 に貼る (10点)

**課題1 の `tarai` と中身は1文字も変えず**、`@memo` を付けたものを
`tarai_memo` という名前で定義せよ。呼び出し回数を数える変数は `MEMO_CALLS` を使う。

(課題1 の `tarai` を残しておきたいので、名前だけ変える)

課題1-b で数えた「異なる引数の組の数」と一致するはず。

### 例題

次のセルを実行して、動きを見る。

`fib` に貼った場合。

In [ ]:
# 例題
@memo
def fib_m(n):
    if n <= 1:
        return n
    return fib_m(n - 1) + fib_m(n - 2)


print(fib_m(100))

大事なのは、再帰の呼び先も**デコレータを貼った名前**にすること。
`fib` を呼んでしまうとキャッシュを通らない。


In [ ]:
MEMO_CALLS = 0


@memo
def tarai_memo(x, y, z):
    global MEMO_CALLS
    MEMO_CALLS += 1
    # TODO: 課題1 と同じ中身を、tarai_memo を呼ぶ形で書く
    pass


import time
MEMO_CALLS = 0
t0 = time.time()
print('tarai_memo(12, 6, 0) =', tarai_memo(12, 6, 0))
print(f'呼び出し回数: {MEMO_CALLS:,} 回 / {time.time() - t0:.4f} 秒')

In [ ]:
# [CHECK] q3b
need('tarai_memo', arity=3)

MEMO_CALLS = 0
tarai_memo.cache.clear()
assert tarai_memo(12, 6, 0) == 12
assert MEMO_CALLS == 225, f'{MEMO_CALLS} 回。@memo が効いていない (課題1 では 12,604,861 回)'
assert MEMO_CALLS == distinct_args(12, 6, 0), '課題1-b の答えと一致するはず'

# メモ化すれば、素朴な実装では絶対に終わらない大きさも一瞬で解ける
MEMO_CALLS = 0
tarai_memo.cache.clear()
assert tarai_memo(100, 50, 0) == 100
# 覚えた組の数そのものを見る。数え方を間違えていても、ここは誤魔化せない
MEMO_CALLS = 0
tarai_memo.cache.clear()
tarai_memo(12, 6, 0)
assert len(tarai_memo.cache) == 225, \
    f'覚えた組が {len(tarai_memo.cache)} 通り。引数の組は 225 通りのはず'
note(f'tarai_memo(12, 6, 0) は {MEMO_CALLS} 回 / 覚えた組 {len(tarai_memo.cache)} 通り')

print(f'OK  tarai_memo(100, 50, 0) は {MEMO_CALLS:,} 回で終わった')


---
## 課題4: コマンド表 (20点)

**定義した瞬間に辞書へ登録する**デコレータを作り、文字列からコマンドを実行できるようにする。

```python
@command('shout')
def shout(text):
    return text.upper() + '!!'

run('shout hello')      # 'HELLO!!'
```

- `command(name)`: **デコレータを返す関数**。`COMMANDS[name]` に登録し、
  **関数そのものはそのまま返す** (包まない)
- `run(line)`: 空白で分け、先頭を命令名、残りを引数として呼ぶ。
  知らない命令なら `ValueError`

**ヒント**: `name, *args = line.split()` で先頭と残りに分けられる。
呼ぶときは `func(*args)`。引数はすべて**文字列**で渡ってくることに注意。

### 例題

次のセルを実行して、動きを見る。

「定義した瞬間に辞書へ登録する」デコレータ。


In [ ]:
# 例題
HANDLERS = {}


def handler(event):
    def deco(func):
        HANDLERS[event] = func
        return func
    return deco


@handler('click')
def on_click(x, y):
    return f'clicked {x},{y}'


print(HANDLERS['click']('1', '2'))   # clicked 1,2

`run` の側は、文字列を切って辞書を引くだけ。


In [ ]:
# 例題
name, *args = 'shout hello world'.split()
print(name, args)    # shout ['hello', 'world']

In [ ]:
COMMANDS = {}


def command(name):
    def deco(func):
        # TODO: COMMANDS[name] に func を入れ、func をそのまま返す
        pass
    return deco


def run(line):
    # TODO: 命令名と引数に分け、COMMANDS から引いて呼ぶ。知らない命令は ValueError
    pass

In [ ]:
@command('calc')
def cmd_calc(op, a, b):
    """2つの整数を演算する"""
    return calc(op, int(a), int(b))


@command('shout')
def cmd_shout(text):
    """大文字にして !! を付ける"""
    return text.upper() + '!!'


@command('rev')
def cmd_rev(text):
    """文字列を逆順にする"""
    return text[::-1]


for line in ['calc * 6 7', 'shout hello', 'rev abc']:
    print(f'{line:16} -> {run(line)}')

In [ ]:
# [CHECK] q4
need('COMMANDS', kind=dict)
need('command', arity=1)
need('run', arity=1)

assert run('calc * 6 7') == 42
assert run('calc ** 2 10') == 1024
assert run('shout hello') == 'HELLO!!'
assert run('rev abc') == 'cba'
assert {'calc', 'shout', 'rev'} <= set(COMMANDS)

# デコレータは関数を包まずそのまま返すこと
assert cmd_shout('hi') == 'HI!!', '@command が関数を壊している'
assert cmd_shout.__name__ == 'cmd_shout'

try:
    run('unknown x')
except ValueError:
    pass
else:
    raise AssertionError('知らない命令は ValueError を出すこと')

# 実行中に命令を増やせること
@command('twice')
def cmd_twice(text):
    """2回くり返す"""
    return text * 2

assert run('twice ab') == 'abab'
print('OK')


---
# 発展課題

余力があれば。


## 発展1: `@trace` で再帰を眺める (5点)

呼び出しの深さだけ字下げして、引数と返り値を表示するデコレータを書け。

```
tarai_t(3, 2, 1)
  tarai_t(2, 2, 1)
  -> 2
  ...
-> 3
```

- 呼び出し時に `関数名(引数...)` を、深さ×2個の空白で字下げして表示
- 返るときに `-> 返り値` を同じ深さで表示
- **返り値は変えないこと**

**ヒント**: 深さを覚える変数を `wrapper` の外に置き、`nonlocal` で書き換える。
`try / finally` にすると、例外が出ても深さが戻る。

### 例題

次のセルを実行して、動きを見る。

深さを覚えて、入るときだけ表示するデコレータ。

In [ ]:
# 例題
def enter_only(func):
    depth = 0

    def wrapper(*args):
        nonlocal depth
        print('  ' * depth + f'{func.__name__}{args}')
        depth += 1
        try:
            return func(*args)
        finally:
            depth -= 1
    return wrapper

課題は、これに「返るときの `-> 返り値`」を足したもの。
返り値を変数で受けてから表示し、最後にその値を返す。


In [ ]:
def trace(func):
    depth = 0

    def wrapper(*args):
        nonlocal depth
        # TODO
        pass

    return wrapper


@trace
def tarai_t(x, y, z):
    if x <= y:
        return y
    return tarai_t(tarai_t(x-1, y, z), tarai_t(y-1, z, x), tarai_t(z-1, x, y))


print('結果:', tarai_t(3, 2, 1))

In [ ]:
# [CHECK] b1
need('trace', arity=1)
need('tarai_t', arity=3)

import io, contextlib
buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    result = tarai_t(3, 2, 1)
out = buf.getvalue()
assert result == 3, '返り値を壊さないこと'
assert out.count('tarai_t') >= 5, '呼び出しが表示されていない'
assert '->' in out, '返り値が表示されていない'
assert any(l.startswith('  ') and not l.startswith('    ')
           for l in out.splitlines()), '深さ1 の行が字下げされていない'
assert any(l.startswith('    ') for l in out.splitlines()), '深さ2 以上の字下げが無い'

# 深さが元に戻ること (2回呼んでも字下げが増えない)
buf2 = io.StringIO()
with contextlib.redirect_stdout(buf2):
    tarai_t(3, 2, 1)
assert buf2.getvalue() == out, '呼ぶたびに深さがずれている'
print('OK')


## 発展2: `help` を自動生成する (5点)

コマンド表に登録された関数の **docstring と引数名**から、使い方一覧を作る
`help_text()` を書け。

```
calc <op> <a> <b>       2つの整数を演算する
rev <text>              文字列を逆順にする
...
```

- 1行につき1コマンド
- 形式は `名前 <引数1> <引数2> ...` のあとに docstring
- **コマンド名は辞書順**に並べる

**ヒント**: `func.__doc__` で docstring、
`inspect.signature(func).parameters` で引数名の一覧が取れる。

**ここが面白いところ**: 一覧を手で書いていない。
**プログラムが自分自身を調べて**説明を作っている。

### 例題

次のセルを実行して、動きを見る。

関数から、名前と説明を取り出す。


In [ ]:
# 例題
import inspect


def introduce(name, times):
    """挨拶する"""
    return ('やあ ' + name) * times


print(list(inspect.signature(introduce).parameters))   # ['name', 'times']
print(introduce.__doc__)                               # 挨拶する

あとは `COMMANDS` を辞書順に回して、1行ずつ組み立てる。


In [ ]:
import inspect


def help_text():
    # TODO
    pass


print(help_text())


In [ ]:
# [CHECK] b2
need('help_text', arity=0)

t = help_text()
assert isinstance(t, str), '文字列を返すこと'
for name in COMMANDS:
    assert name in t, f'{name} が一覧に無い'
assert '<op>' in t and '<text>' in t, '引数名を <> で囲んで並べること'
assert '逆順' in t, 'docstring を使うこと'
assert t.index('calc') < t.index('rev') < t.index('shout'), '辞書順に並べること'
assert len(t.splitlines()) >= len(COMMANDS), '1行につき1コマンド'
print('OK')


---
## 自己確認

`score()` で全部まとめて確かめる。何度やってもよい。
`NG` の行に、次に何をすればいいかが出る。


In [ ]:
# [SKIP] 採点の対象外
score()


---
## 提出

提出は kibaco から。`score()` と採点ページは下書き確認用。

1. 上から実行し、出力を残す
2. `ファイル → .ipynb をダウンロード`
3. 締切までに kibaco へ出す

確認セルが全部 `OK` なら提出してよい。確認セルを消しても教員の採点は別の検査で行う。
